# Data Cleaning and Sorting
Cleans and sorts `DATA.csv` using **pandas** and **numpy**.

**Steps:**
1. Load data
2. Remove duplicates
3. Trim whitespace & replace pseudo-nulls
4. Handle missing values
5. Standardize date format
6. Validate numeric columns
7. Sort by Date and OrderID
8. Save cleaned output

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# ── 1. Load ──────────────────────────────────────────────────────────────────
file_path = 'DATA.csv'
try:
    df = pd.read_csv(file_path)
    print(f"Loaded {len(df):,} rows × {len(df.columns)} columns")
except FileNotFoundError:
    raise FileNotFoundError(f"'{file_path}' not found. Make sure it is in the same folder as this notebook.")

# ── 2. Remove duplicates ─────────────────────────────────────────────────────
before = len(df)
df.drop_duplicates(inplace=True)
print(f"Duplicates removed : {before - len(df)}")

# ── 3. Trim whitespace & replace pseudo-nulls ─────────────────────────────────
PSEUDO_NULLS = ['nan', 'null', 'none', 'n/a', 'na', '?', '']
str_cols = df.select_dtypes(include='object').columns
for col in str_cols:
    df[col] = df[col].astype(str).str.strip()
    df[col] = np.where(df[col].str.lower().isin(PSEUDO_NULLS), np.nan, df[col])

# ── 4. Handle missing values ─────────────────────────────────────────────────
df.dropna(subset=['OrderID', 'Date', 'CustomerID'], inplace=True)
for col in ['Product', 'PaymentMethod', 'OrderStatus', 'CouponCode', 'ReferralSource']:
    if col in df.columns:
        df[col] = df[col].fillna('Unknown')
print(f"Missing values handled. Rows remaining: {len(df):,}")

# ── 5. Standardise Date → YYYY-MM-DD ─────────────────────────────────────────
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
invalid_dates = df['Date'].isna().sum()
if invalid_dates:
    print(f"Dropping {invalid_dates} rows with unparseable dates")
    df.dropna(subset=['Date'], inplace=True)
df['Date'] = df['Date'].dt.strftime('%Y-%m-%d')

# ── 6. Validate & recalculate numeric columns ─────────────────────────────────
df['Quantity']   = pd.to_numeric(df['Quantity'],  errors='coerce').fillna(0).astype(int)
df['UnitPrice']  = pd.to_numeric(df['UnitPrice'], errors='coerce').fillna(0.0).round(2)
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)].copy()
df['TotalPrice'] = (df['Quantity'] * df['UnitPrice']).round(2)
print(f"Numeric validation done. Rows remaining: {len(df):,}")

# ── 7. Sort ───────────────────────────────────────────────────────────────────
df.sort_values(by=['Date', 'OrderID'], ascending=True, inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"Sorted by Date ↑ then OrderID ↑")

# ── 8. Save ───────────────────────────────────────────────────────────────────
output_path = 'DATA_cleaned.csv'
df.to_csv(output_path, index=False)
print(f"\nSaved → '{output_path}'  ({len(df):,} rows)")

In [ ]:
# Preview
df.head(10)

In [ ]:
# Quick summary stats
df.describe()